# 17. Type Hinting, Generics & Modern Dataclasses: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **17. Type Hinting, Generics & Modern Dataclasses**. Python's static typing ecosystem (`typing` module) enables type checking via tools like `mypy` and IDE autocompletion. This notebook covers type annotations, union types (`X | Y`), optional types, generic type variables (`TypeVar`, `Generic`), protocols (`Protocol` for structural subtyping), and modern `@dataclass` features (`slots=True`, `frozen=True`, `field(default_factory=...)`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Type Annotation: `Union[A, B]`
- [x] 🔹 Type Annotation: `Optional[T]`
- [x] 🔹 Type Annotation: `Callable[[Args], Return]`
- [x] 🔹 Generic Containers: `List[T]` & `Dict[K, V]`
- [x] 🔹 Generic Type Variables: `TypeVar`
- [x] 🔹 Structural Typing with `Protocol` (Static Duck Typing)
- [x] 🔹 Data Modeling: `@dataclass`
- [x] 🔹 Immutable Data Modeling: `@dataclass(frozen=True)`
- [x] 🔹 Value Constraints: `Literal`
- [x] 🔹 Constant Protection: `Final`
- [x] 🔹 Dictionary Schema: `TypedDict`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Type Annotation: `Union[A, B]`
- **What it does:** Annotates parameters or return types that can accept multiple distinct candidate types.
- **Syntax:** `Union[A, B]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.
- **Dataset Application & Code Demonstration:** Applies Type Annotation on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [2]:
def parse_num(val: Union[int, float, str]) -> float:
    return float(val)

print('Union parameter parsed:', parse_num(transactions[0]['transaction_amount']))

Union parameter parsed: 607.78


### 🔹 Type Annotation: `Optional[T]`
- **What it does:** Shorthand for `Union[T, None]`, representing optional arguments that can be None.
- **Syntax:** `Optional[T]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.
- **Dataset Application & Code Demonstration:** Applies Type Annotation on fintech records using columns `card_type`, `transaction_id` to demonstrate real-world execution.


In [3]:
def find_transaction(tx_id: str) -> Optional[dict]:
    return transactions[0] if tx_id == transactions[0]['transaction_id'] else None

print('Optional return:', find_transaction('TX110686')['card_type'] if find_transaction('TX110686') else 'None')

Optional return: None


### 🔹 Type Annotation: `Callable[[Args], Return]`
- **What it does:** Annotates higher-order functions expecting callback functions as arguments.
- **Syntax:** `Callable[[Args], Return]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.
- **Dataset Application & Code Demonstration:** Demonstrates Type Annotation with practical fintech data structures and variables in the following code block.


In [4]:
def apply_fee(fee_calc: Callable[[float], float], amt: float) -> float:
    return fee_calc(amt)

print('Callable output:', apply_fee(lambda a: a * 0.02, 500.0))

Callable output: 10.0


### 🔹 Generic Containers: `List[T]` & `Dict[K, V]`
- **What it does:** Annotates collection element types for static type analysis.
- **Syntax:** `List[T]`
- **Key Note:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.
- **Dataset Application & Code Demonstration:** Applies Generic Containers on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [5]:
def extract_ids(records: List[Dict[str, str]]) -> List[str]:
    return [r['transaction_id'] for r in records]

print('Generic annotated extraction:', extract_ids(transactions[:3]))

Generic annotated extraction: ['TX109326', 'TX106376', 'TX103301']


### 🔹 Generic Type Variables: `TypeVar`
- **What it does:** Defines type variables enabling generic reusable functions preserving input types.
- **Syntax:** `TypeVar`
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.
- **Dataset Application & Code Demonstration:** Applies Generic Type Variables on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [6]:
T = TypeVar('T')
def first_item(items: List[T]) -> T:
    return items[0]

print('Generic TypeVar output:', first_item(transactions[:3])['transaction_id'])

Generic TypeVar output: TX109326


### 🔹 Structural Typing with `Protocol` (Static Duck Typing)
- **What it does:** Defines structural interfaces where classes conform implicitly without inheritance.
- **Syntax:** `Protocol`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Demonstrates Structural Typing with `Protocol` (Static Duck Typing) with practical fintech data structures and variables in the following code block.


In [7]:
class Payable(Protocol):
    def get_amount(self) -> float: ...

class Invoice:
    def __init__(self, amt): self.amt = amt
    def get_amount(self) -> float: return self.amt

def process_payment(item: Payable) -> str:
    return f'Processed payment for ${item.get_amount():,.2f}'

print(process_payment(Invoice(1500.0)))

Processed payment for $1,500.00


### 🔹 Data Modeling: `@dataclass`
- **What it does:** Generates `__init__`, `__repr__`, `__eq__` boilerplate automatically based on type annotations.
- **Syntax:** `@dataclass`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.
- **Dataset Application & Code Demonstration:** Applies Data Modeling on fintech records using columns `card_type`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [8]:
@dataclass
class TxData:
    id: str
    amount: float
    card: str

tx_inst = TxData(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount']), transactions[0]['card_type'])
print('Dataclass instance:', tx_inst)

Dataclass instance: TxData(id='TX109326', amount=607.78, card='Visa')


### 🔹 Immutable Data Modeling: `@dataclass(frozen=True)`
- **What it does:** Enforces immutability: instance attribute mutation raises `FrozenInstanceError`.
- **Syntax:** `@dataclass(frozen=True)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.
- **Dataset Application & Code Demonstration:** Demonstrates Immutable Data Modeling with practical fintech data structures and variables in the following code block.


In [9]:
@dataclass(frozen=True)
class FrozenTx:
    id: str
    amount: float

frozen_inst = FrozenTx('TX1', 100.0)
print('Frozen dataclass:', frozen_inst)

Frozen dataclass: FrozenTx(id='TX1', amount=100.0)


### 🔹 Value Constraints: `Literal`
- **What it does:** Restricts variable or parameter to exact literal values.
- **Syntax:** `Literal`
- **Key Note:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.
- **Dataset Application & Code Demonstration:** Demonstrates Value Constraints with practical fintech data structures and variables in the following code block.


In [10]:
def set_card(card: Literal['Visa', 'MasterCard', 'Amex', 'Discover']):
    return f'Card set to {card}'

print(set_card('Visa'))

Card set to Visa


### 🔹 Constant Protection: `Final`
- **What it does:** Declares constants that static type checkers ensure are never reassigned or overridden.
- **Syntax:** `Final`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Demonstrates Constant Protection with practical fintech data structures and variables in the following code block.


In [11]:
CURRENCY: Final[str] = 'USD'
print('Final constant:', CURRENCY)

Final constant: USD


### 🔹 Dictionary Schema: `TypedDict`
- **What it does:** Defines expected dictionary key names and value types for static validation.
- **Syntax:** `TypedDict`
- **Key Note:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.
- **Dataset Application & Code Demonstration:** Applies Dictionary Schema on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [12]:
class TransactionSchema(TypedDict):
    transaction_id: str
    transaction_amount: float

schema_obj: TransactionSchema = {
    'transaction_id': transactions[0]['transaction_id'],
    'transaction_amount': float(transactions[0]['transaction_amount'])
}
print('TypedDict schema object:', schema_obj)

TypedDict schema object: {'transaction_id': 'TX109326', 'transaction_amount': 607.78}


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Nominal vs Structural Subtyping in Python
- **Objective:** Q1: Nominal vs Structural Subtyping in Python
- **Approach:** Explain difference between Nominal typing (`isinstance` / ABCs) and Structural typing (`typing.Protocol` / duck typing).
- **Syntax:** `isinstance` vs `Protocol`

In [13]:
print('Nominal Subtyping: Explicit inheritance hierarchy (isinstance).')
print('Structural Subtyping: Shape and method signature matching (Protocol).')

Nominal Subtyping: Explicit inheritance hierarchy (isinstance).
Structural Subtyping: Shape and method signature matching (Protocol).
